# Lesson 0: Setup

**Why:** Before we build anything, we make sure the tools work. We need Python, PyTorch (the library that does the heavy math for us), and the text our model will learn from.

**How to run a cell:** click inside it and press **Shift + Enter**. Before each cell, answer the prediction question in the chat first, then run it and compare.

## 1. Is PyTorch installed?

In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu


## 2. Which device will do the math?

- **CPU:** your laptop's main processor. It has a few strong cores that are good at anything.
- **GPU (CUDA):** an NVIDIA graphics card. It has thousands of weak cores that are great at doing the same multiplication many times in parallel, which is exactly what neural networks need.

PyTorch spreads its work over several **threads** (parallel lanes of work on the CPU).

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Training will run on:", device)
print("CPU threads PyTorch will use:", torch.get_num_threads())

Training will run on: cpu
CPU threads PyTorch will use: 10


## 3. How fast is your laptop at the core operation of a GPT?

Almost everything a GPT does is **matrix multiplication**: many rows of numbers multiplied against many columns.

Multiplying two 1000 × 1000 grids of numbers takes 1000 × 1000 × 1000 = **1 billion** multiply-and-add steps.

(`torch.randn` makes a grid of random numbers, and `@` means matrix multiply. Lesson 1 covers both properly.)

In [3]:
import time

a = torch.randn(1000, 1000)
b = torch.randn(1000, 1000)

a @ b  # warm-up run (the first one is always slower)

start = time.perf_counter()
for _ in range(10):
    c = a @ b
seconds = (time.perf_counter() - start) / 10

print(f"One 1000x1000 matrix multiply: {seconds * 1000:.1f} ms")
print(f"Speed: about {2 * 1000**3 / seconds / 1e9:.0f} billion math operations per second")

One 1000x1000 matrix multiply: 13.7 ms
Speed: about 146 billion math operations per second


## 4. The dataset: Nikola Tesla

*The Inventions, Researches and Writings of Nikola Tesla* (1894). It collects Tesla's lectures and patents, with linking chapters by the editor, Thomas Commerford Martin. Our GPT will read it character by character and learn to write text that sounds like it.

(The file is made by `src/tinygpt/download_tesla.py`. It downloads the book from Project Gutenberg and removes the license text, the index and the picture captions.)

In [4]:
from pathlib import Path

text = Path("../data/input.txt").read_text(encoding="utf-8")

print(f"Characters in the file: {len(text):,}")
print("-" * 40)
print(text[:300])

Characters in the file: 976,605
----------------------------------------
THE INVENTIONS
  RESEARCHES AND WRITINGS

  OF

  NIKOLA TESLA


  WITH SPECIAL REFERENCE TO HIS WORK IN POLYPHASE
  CURRENTS AND HIGH POTENTIAL LIGHTING


  BY

  THOMAS COMMERFORD MARTIN

  Editor THE ELECTRICAL ENGINEER; Past-President American Institute
  Electrical Engineers


  1894
  THE ELEC


How many **different** characters does all that text use? Count letters, capitals, punctuation, space and newline.

(`set(text)` keeps one copy of each character and drops the duplicates. `sorted` puts them in order.)

In [5]:
unique_chars = sorted(set(text))

print(f"Different characters used: {len(unique_chars)}")
print(repr("".join(unique_chars)))

Different characters used: 85
'\n !"&\'()*+,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]^abcdefghijklmnopqrstuvwxyz{}'


## 5. Final check

In [6]:
assert torch.__version__, "PyTorch is missing"
assert len(text) > 900_000, "The dataset looks incomplete"

print("Setup complete. Ready for Lesson 1.")

Setup complete. Ready for Lesson 1.
